<a href="https://colab.research.google.com/github/DSC-SPIDAL/mlcommons/blob/main/Please_put_the_ideas_of_V39_in_UDR_https_github_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is an excellent request. Integrating the optimized, iterative engine of V39 into the architectural philosophy of UniversalDeepResearch (UDR) while maintaining Colab compatibility requires a significant restructuring.

UDR emphasizes modularity, separation of concerns, and a clear distinction between the research strategy and the execution engine. V39, while efficient, bundled these concepts into a single pipeline class.

To achieve this integration, I have developed **DeepCollector V40**.

### V40: UDR-Inspired Architecture, V39 Optimized Engine

V40 adopts a modular architecture inspired by UDR, organizing the V39 logic into distinct functional components. This makes the system more maintainable, auditable, and adaptable, reflecting UDR's design principles.

**V40 Architectural Components:**

1.  **Configuration (`v40_config.py`):** Centralized setup and dependency management.
2.  **Tools Layer (`v40_tools/`):** Dedicated modules for external interactions, adhering to the principle of modular tools.
      * `llm_utils.py`: Centralized Gemini model management (Pro and Flash) and LlamaIndex configuration.
      * `search.py`: Google Search implementation.
      * `loader.py`: Optimized Document Loader (retaining V39's hashing and incremental loading).
      * `kb.py`: External Knowledge Base access.
3.  **Core Logic (`v40_core/`):** The specialized research components.
      * `schema.py`: Defines the target data structure.
      * `analyzer.py`: Handles Stage 1 (Identification) and Stage 2 (Classification/Filtering).
      * `extractor.py`: Manages Stage 3 (Precision RAG).
      * `orchestrator.py`: The central engine. This component manages the research state and executes the iterative loop, incorporating the **V39 optimization engine** (Iterative Refinement, Incremental Scanning, Confidence Locking).

This architecture successfully integrates the efficiency of V39 within a robust, UDR-inspired software structure.

Here is the complete V40 code:

In [ ]:
# ====================================================================================
# DeepCollector V40 (Portable Single Script Version)
# UDR-Inspired Architecture, V39 Optimized Engine
# ====================================================================================
import sys
import os
import importlib
import subprocess
import traceback
import shutil

# ------------------------------------------------------------------------------------
# 1. Install Dependencies
# ------------------------------------------------------------------------------------
print("--- Installing/Upgrading Dependencies ---")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "google-generativeai", "gspread", "tenacity",
                           "beautifulsoup4", "llama-index", "llama-index-readers-file",
                           "llama-index-llms-google-genai",
                           "llama-index-embeddings-google-genai", "pypdf", "pandas", "requests", "--upgrade"])
    print("Dependencies installed successfully.")
except Exception as e:
    print(f"*** ERROR installing dependencies: {e} ***")

# ------------------------------------------------------------------------------------
# 2. Define Module Contents as Strings
# ------------------------------------------------------------------------------------

# ====================================================================================
# CONFIGURATION
# ====================================================================================

# --- v40_config.py ---
V40_CONFIG_PY = """
# v40_config.py
# Configuration, Setup, and Global Dependencies for DeepCollector V40.

import pandas as pd
import requests
import json
import time
import re
import io
import os
import hashlib
import pickle
import math
from IPython.display import display
import google.generativeai as genai
import gspread

from pathlib import Path
from tenacity import retry, stop_after_attempt, wait_exponential, RetryError, retry_if_exception_type
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin

# Import Google API exceptions
try:
    from google.api_core import exceptions as google_exceptions
except ImportError:
    google_exceptions = None

# LlamaIndex Core Imports
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import NodeWithScore

# Imports specific to Google Colab authentication and Drive mounting
try:
    from google.colab import auth, userdata, drive
    from google.auth import default
except ImportError:
    auth = userdata = drive = None
    default = None

# ------------------------------------------------------
# Global Configuration Variables (EDIT THESE FOR NEW PROJECTS)
# ------------------------------------------------------

# Define the Target URL(s)
PRIMARY_URLS = [
    "http://arxiv.org/abs/2310.04948",
    "https://github.com/DC-research/TEMPO"
    ]
PROJECT_NAME_HINT = "TEMPO"
DOMAIN_CONTEXT = "Time Series Forecasting/Analysis"

# V40: Updated Spreadsheet Name
SPREADSHEET_NAME = "Tempo_Catalog_V40_UDR_Arch_V39_Engine"
MAX_RESEARCH_ITERATIONS = 5

# Confidence Auditing Thresholds (V39 Optimization)
CONFIDENCE_THRESHOLD = 0.75
CONFIDENCE_LOCK_THRESHOLD = 0.95 # Cells above this are skipped in subsequent iterations

# Generalized Knowledge Base (GKB) Configuration
GENERAL_KNOWLEDGE_PKL_PATH = "/content/drive/My Drive/AIResults/Catalogs/project_info_cache.pkl"

# Generalized Model List (used for filtering)
KNOWN_TS_MODELS = {
    "PatchTST", "TimesNet", "iTransformer", "DLinear", "Autoformer",
    "Informer", "Transformer", "LSTM", "GRU", "TCN", "DeepAR", "N-BEATS",
    "TEMPO"
}

# Debug Mode
DEBUG_MODE = False

# ------------------------------------------------------
# Initialization Function (Environment Setup)
# ------------------------------------------------------

def initialize_environment():
    \"\"\"Handles authorization, API configuration, and model initialization.\"\"\"
    # 1. Google Drive Authorization
    print("\\n--- Google Drive Authorization (for Knowledge Base Access) ---")
    if drive:
        try:
            if not os.path.exists('/content/drive/My Drive'):
                print("CRITICAL: A prompt may appear requesting access to Google Drive.")
                drive.mount('/content/drive', force_remount=True)
            print("Google Drive accessible.")
        except Exception as e:
            print(f"\\n*** WARNING: Google Drive mounting failed: {e}. Knowledge Base Tool will be disabled. ***")
    else:
        print("Google Colab 'drive' library not found. Skipping Drive mount.")

    # 2. API Configuration
    GEMINI_API_KEY = None
    GOOGLE_SEARCH_API_KEY = None
    GOOGLE_CSE_ID = None
    SEARCH_ENABLED = False

    print("\\n--- API Configuration Check ---")
    try:
        if userdata:
            GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
            if not GEMINI_API_KEY:
                raise ValueError("GEMINI_API_KEY not found in Colab Secrets.")
            genai.configure(api_key=GEMINI_API_KEY)
            print("Gemini API configured.")

            GOOGLE_SEARCH_API_KEY = userdata.get('GOOGLE_SEARCH_API_KEY')
            GOOGLE_CSE_ID = userdata.get('GOOGLE_CSE_ID')

            if GOOGLE_SEARCH_API_KEY and GOOGLE_CSE_ID:
                SEARCH_ENABLED = True
                print("Google Search API configured. Dynamic Discovery ENABLED.")
            else:
                print("\\n*** WARNING: Google Search API credentials missing. Dynamic Discovery DISABLED. ***\\n")
        else:
            print("Colab userdata not available.")
    except Exception as e:
        print(f"Configuration Error: {e}")

    return {
        "GEMINI_API_KEY": GEMINI_API_KEY,
        "SEARCH_ENABLED": SEARCH_ENABLED,
        "GOOGLE_SEARCH_API_KEY": GOOGLE_SEARCH_API_KEY,
        "GOOGLE_CSE_ID": GOOGLE_CSE_ID
    }

# ------------------------------------------------------
# Retry Strategies (Global)
# ------------------------------------------------------

# Define robust retry strategy (Gemini, Search, Requests)
try:
    retry_conditions = retry_if_exception_type(requests.exceptions.ConnectionError)
except AttributeError:
    retry_conditions = retry_if_exception_type(Exception) # Fallback if requests is not fully initialized

if google_exceptions:
    retry_conditions = (
        retry_conditions |
        retry_if_exception_type(google_exceptions.ResourceExhausted) |
        retry_if_exception_type(google_exceptions.InternalServerError) |
        retry_if_exception_type(google_exceptions.ServiceUnavailable)
    )

RETRY_STRATEGY = retry(
    wait=wait_exponential(multiplier=1, min=4, max=60),
    stop=stop_after_attempt(6),
    retry=retry_conditions,
    before_sleep=lambda retry_state: print(f"    [API Retry] Retrying {retry_state.fn.__name__} (Attempt {retry_state.attempt_number})...")
)

# Define specific retry strategy for GSpread operations (Sheets API)
GSPREAD_RETRY_STRATEGY = retry(
    wait=wait_exponential(multiplier=2, min=5, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(gspread.exceptions.APIError) | retry_if_exception_type(Exception),
    before_sleep=lambda retry_state: print(f"    [GSpread Retry] Retrying {retry_state.fn.__name__} (Attempt {retry_state.attempt_number})...")
)
"""

# ====================================================================================
# TOOLS (External Interactions)
# ====================================================================================

# --- v40_tools/llm_utils.py ---
V40_TOOLS_LLM_UTILS_PY = """
# v40_tools/llm_utils.py
# Centralized LLM (Gemini) interaction, configuration, and utilities.

import google.generativeai as genai
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter

# V40: Import LlamaIndex integrations here, as they are part of the LLM tooling
try:
    from llama_index.llms.google_genai import GoogleGenAI
    from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
except ImportError as e:
    print(f"\\n*** CRITICAL IMPORT ERROR: {e} ***\\n")
    GoogleGenAI = GoogleGenAIEmbedding = None

# Import global configuration and retry strategy
from v40_config import RETRY_STRATEGY

# Global model instances
MODEL_FAST = None
MODEL_ACCURATE = None

@RETRY_STRATEGY
def generate_content_with_retry(model, prompt):
    \"\"\"Robust wrapper for Gemini API calls.\"\"\"
    if model is None:
        raise ValueError("Model not initialized.")
    try:
        return model.generate_content(prompt)
    except Exception as e:
        raise

def initialize_models(api_key):
    \"\"\"Initializes Gemini models and configures LlamaIndex settings.\"\"\"
    global MODEL_FAST, MODEL_ACCURATE

    if not api_key or not GoogleGenAI or not GoogleGenAIEmbedding:
        print("Models cannot be initialized. Missing API key or dependencies.")
        return

    # Agentic Control Flow Models (Direct API)
    MODEL_FAST = genai.GenerativeModel(
        'gemini-1.5-flash-latest',
        generation_config={"response_mime_type": "application/json"}
    )
    # V40 retains Pro for accuracy in critical steps
    MODEL_ACCURATE = genai.GenerativeModel(
        'gemini-1.5-pro-latest',
        generation_config={"response_mime_type": "application/json"}
    )

    # LlamaIndex RAG Models
    llm_rag = GoogleGenAI(model="models/gemini-1.5-pro-latest", api_key=api_key)
    embed_model = GoogleGenAIEmbedding(model_name="models/embedding-001", api_key=api_key)

    # Configure LlamaIndex global settings
    Settings.llm = llm_rag
    Settings.embed_model = embed_model
    Settings.embed_batch_size = 100
    Settings.node_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=200)
    Settings.show_progress = True
    print("Initialized Gemini Control Flow and LlamaIndex RAG components.")

def get_model(accuracy_level='accurate'):
    \"\"\"Provides access to the initialized models.\"\"\"
    if accuracy_level == 'fast':
        return MODEL_FAST
    else:
        return MODEL_ACCURATE
"""

# --- v40_tools/search.py ---
V40_TOOLS_SEARCH_PY = """
# v40_tools/search.py
# Google Custom Search Tool implementation.

import requests
from v40_config import RETRY_STRATEGY, DEBUG_MODE

class SearchTool:
    def __init__(self, api_key, cse_id):
        self.api_key = api_key
        self.cse_id = cse_id
        self.base_url = "https://www.googleapis.com/customsearch/v1"

    @RETRY_STRATEGY
    def search(self, query, num_results=10):
        if DEBUG_MODE:
            print(f"  [Search Tool] Executing query: '{query}'")
        params = {
            'q': query,
            'key': self.api_key,
            'cx': self.cse_id,
            'num': num_results
        }
        try:
            response = requests.get(self.base_url, params=params)
            response.raise_for_status()
            results = response.json()

            formatted_results = []
            if 'items' in results:
                for item in results['items']:
                    formatted_results.append({
                        "snippet": item.get('snippet', ''),
                        "url": item.get('link', ''),
                        "source_title": item.get('title', '')
                    })
            return formatted_results
        except Exception as e:
            print(f"    [Search Tool Error] Error during search: {e}")
            return []
"""

# --- v40_tools/loader.py ---
V40_TOOLS_LOADER_PY = """
# v40_tools/loader.py
# Generalized Document Loader (Web, PDF, GitHub).

import requests
import hashlib
import os
from urllib.parse import urlparse
from pathlib import Path
from llama_index.core import Document

# Import global configuration and retry strategy
from v40_config import RETRY_STRATEGY, DEBUG_MODE

# V40: Import LlamaIndex readers here, as they are part of the loading tooling
try:
    from llama_index.readers.file import PDFReader, HTMLTagReader, FlatReader
except ImportError as e:
    print(f"\\n*** CRITICAL IMPORT ERROR: {e} ***\\n")
    PDFReader = HTMLTagReader = FlatReader = None

class DocumentLoader:
    def __init__(self):
        self.headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        self.timeout = 25
        self.loaded_content_hashes = set()

        if PDFReader and HTMLTagReader and FlatReader:
            self.pdf_reader = PDFReader()
            self.html_reader = HTMLTagReader(tag="body")
            self.default_reader = FlatReader()
            self.initialized = True
        else:
            self.initialized = False

    def _validate_url(self, url):
        if not isinstance(url, str): return False
        try:
            result = urlparse(url)
            return all([result.scheme, result.netloc]) and result.scheme in ['http', 'https']
        except:
            return False

    def _generate_content_hash(self, content_bytes):
        return hashlib.sha256(content_bytes[:51200]).hexdigest()

    @RETRY_STRATEGY
    def _fetch_url(self, url, headers, timeout):
       return requests.get(url, headers=headers, timeout=timeout)

    def load(self, url):
        if not self.initialized: return None
        if not self._validate_url(url): return None
        if url.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.zip', '.tar.gz', '.npy')): return None

        if DEBUG_MODE:
            print(f"Attempting to load document from: {url}")

        # Handle specific patterns
        if "github.com" in url:
            return self._load_github_readme(url)

        # Standardize ArXiv links
        original_url = url
        if "arxiv.org" in url:
            url = url.replace("/abs/", "/pdf/").replace("http://", "https://")
            if not url.lower().endswith(".pdf"):
                url += ".pdf"

        temp_file_path = None
        try:
            response = self._fetch_url(url, self.headers, self.timeout)
            response.raise_for_status()

            content_bytes = response.content
            content_hash = self._generate_content_hash(content_bytes)

            if content_hash in self.loaded_content_hashes:
                if DEBUG_MODE:
                    print(f"  [Duplicate] Content already loaded. Skipping.")
                return []

            # Content-Type Detection
            content_type = response.headers.get('Content-Type', '')
            temp_dir = "/tmp/llama_downloads"
            os.makedirs(temp_dir, exist_ok=True)

            if 'application/pdf' in content_type:
                ext = ".pdf"; reader = self.pdf_reader
            elif 'text/html' in content_type:
                ext = ".html"; reader = self.html_reader
            elif url.lower().endswith('.pdf') and 'text/html' not in content_type:
                ext = ".pdf"; reader = self.pdf_reader
            else:
                ext = ".txt"; reader = self.default_reader

            # ArXiv correction if HTML returned
            if "arxiv.org" in original_url and ext == ".pdf" and content_bytes.strip().startswith(b'<!DOC'):
                print(f"  [Loader Correction] ArXiv URL returned HTML despite PDF request. Switching parser.")
                ext = ".html"; reader = self.html_reader

            temp_file_path = os.path.join(temp_dir, f"doc_{content_hash}{ext}")

            # Write content to temp file
            if ext == '.html' or ext == '.txt':
                try:
                    text_content = content_bytes.decode('utf-8', errors='ignore')
                except UnicodeDecodeError:
                    text_content = str(content_bytes)
                with open(temp_file_path, 'w', encoding='utf-8') as f:
                    f.write(text_content)
            else:
                with open(temp_file_path, 'wb') as f:
                    f.write(content_bytes)

            # Attempt to load documents
            try:
                documents = reader.load_data(file=Path(temp_file_path))
            except Exception as load_e:
                print(f"  [Loader Error] Failed to parse document content from {url} (Type: {ext}). Error: {load_e}")
                return None

            for doc in documents:
                doc.metadata["source_url"] = original_url

            self.loaded_content_hashes.add(content_hash)
            return documents

        except Exception as e:
            return None
        finally:
            if temp_file_path and os.path.exists(temp_file_path):
                try:
                    os.remove(temp_file_path)
                except OSError:
                    pass

    @RETRY_STRATEGY
    def _fetch_github(self, url, headers, timeout):
        return requests.get(url, headers=headers, timeout=timeout)

    def _load_github_readme(self, repo_url):
        try:
            repo_url = repo_url.strip('/')
            if repo_url.lower().endswith('.git'):
                repo_url = repo_url[:-4]

            parts = repo_url.split('/')
            try:
                user_index = [i for i, part in enumerate(parts) if 'github.com' in part][0] + 1
                user, repo = parts[user_index], parts[user_index+1]
            except (IndexError):
                return None

            for branch in ['main', 'master']:
                raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/README.md"
                response = self._fetch_github(raw_url, self.headers, self.timeout)

                if response.status_code == 200:
                    content_bytes = response.content
                    content_hash = self._generate_content_hash(content_bytes)

                    if content_hash in self.loaded_content_hashes:
                        return []

                    self.loaded_content_hashes.add(content_hash)
                    return [Document(text=response.text, metadata={"source_url": repo_url})]
        except Exception as e:
            pass
        return None
"""

# --- v40_tools/kb.py ---
V40_TOOLS_KB_PY = """
# v40_tools/kb.py
# Generalized Knowledge Base (GKB) Tool.

import os
import pickle
import pandas as pd
import re

class KnowledgeBaseTool:
    def __init__(self, pkl_path):
        self.pkl_path = pkl_path
        self.kb_df = self._load_knowledge_base()
        if self.kb_df is not None:
            print(f"KnowledgeBaseTool initialized with {len(self.kb_df)} entries.")
        else:
            print("KnowledgeBaseTool initialized (empty).")

    def _load_knowledge_base(self):
        \"\"\"Loads and parses the Pickle file from Google Drive.\"\"\"
        if not self.pkl_path or not os.path.exists(self.pkl_path):
            # Suppress warning if Drive isn't mounted
            if os.path.exists('/content/drive/My Drive') or (self.pkl_path and not self.pkl_path.startswith('/content/drive')):
                 print(f"  [KB Tool] WARNING: KB file not found at {self.pkl_path}.")
            return None

        try:
            with open(self.pkl_path, 'rb') as f:
                loaded_data = pickle.load(f)

            if isinstance(loaded_data, dict) and 'tables' in loaded_data and isinstance(loaded_data['tables'], list):
                if not loaded_data['tables']:
                    return None
                df_knowledge = pd.concat(loaded_data['tables'], ignore_index=True)
                return df_knowledge
            elif isinstance(loaded_data, pd.DataFrame):
                return loaded_data
            else:
                print("  [KB Tool] ERROR: Pickle file structure is not recognized.")
                return None
        except Exception as e:
            print(f"  [KB Tool] ERROR: Failed to load or process KB file. Error: {e}")
            return None

    def lookup(self, dataset_name):
        \"\"\"Searches the KB DataFrame for a matching dataset name.\"\"\"
        if self.kb_df is None or not dataset_name:
            return None

        name_columns = ['Name', 'Project', 'Collection', 'Dataset Name', 'Dataset']

        for col in name_columns:
            if col in self.kb_df.columns:
                try:
                    match = self.kb_df[self.kb_df[col].str.lower() == dataset_name.lower()]
                    if not match.empty:
                        return match.iloc[0].to_dict()
                except AttributeError:
                    continue
        return None

    def extract_kb_value(self, kb_entry, keys):
        \"\"\"Helper to extract a value from a KB entry using prioritized keys.\"\"\"
        if kb_entry is None:
            return None

        for key in keys:
            if key in kb_entry and pd.notna(kb_entry[key]):
                val = str(kb_entry[key])
                # Clean potential HYPERLINK formulas, prioritizing the URL
                if val.lower().startswith('=hyperlink('):
                    match_url = re.search(r'"(https?://[^"]+)"', val, re.IGNORECASE)
                    if match_url:
                        return match_url.group(1)
                return val
        return None
"""

# ====================================================================================
# CORE LOGIC
# ====================================================================================

# --- v40_core/schema.py ---
V40_CORE_SCHEMA_PY = """
# v40_core/schema.py
# Defines the target data structure and associated queries.

# Define the schema and corresponding targeted queries
CATALOG_SCHEMA_DEFINITION = {
    "Domain": {
        "dtype": "string",
        "query": "What is the domain or area of application for the {name} dataset (e.g., Finance, Healthcare, Energy)?",
        "keywords": "domain area application"
    },
    "Number of Variables at each time point": {
        "dtype": "string or number",
        "query": "What is the dimensionality or number of variables at each time point for the {name} dataset? If it varies, provide the exact range (min-max).",
        "keywords": "dimensionality variables count features"
    },
    "Number of Time Points": {
        "dtype": "string or number",
        "query": "What is the length or total number of time points for the {name} dataset? If it varies, provide the exact range (min-max).",
        "keywords": "length time points samples"
    },
    "Time interval between points": {
        "dtype": "string",
        "query": "What is the frequency or time interval between points for the {name} dataset (e.g., 1 hour, 15 minutes)?",
        "keywords": "frequency interval resolution"
    },
    "Original Source or Originator of Data": {
        "dtype": "string (RAW URL or plain text)",
        "query": "Who is the original creator or organization that originated the {name} dataset? Provide their official website URL if available.",
        "keywords": "original creator originator official website institution"
    },
    "Link to Data": {
        "dtype": "string (RAW URLs separated by '; ')",
        "query": "What are all the useful URLs (repository, direct download link, HuggingFace) for accessing the {name} dataset? List all found.",
        "keywords": "download link URL repository access"
    },
    "Detailed Description": {
        "dtype": "string (Strictly Single line only)",
        "query": "Provide a detailed, single-line description of the {name} dataset, its contents, and its context.",
        "keywords": "description summary details"
    }
}

MISSING_DATA_PLACEHOLDERS = {
    "", "[missing]", "unknown", "n/a", "not found", "not specified",
    "not available", "not mentioned", "none", "missing"
}

AMBIGUOUS_DATASETS = {"traffic", "weather", "ecl", "electricity", "exchange", "illness", "finance"}
"""

# --- v40_core/analyzer.py ---
V40_CORE_ANALYZER_PY = """
# v40_core/analyzer.py
# Implements Stage 1 (Identification) and Stage 2 (Classification/Filtering).

import json
from tenacity import RetryError

# Import centralized LLM utilities
from v40_tools.llm_utils import get_model, generate_content_with_retry

# Import configuration and schema
from v40_config import DOMAIN_CONTEXT, KNOWN_TS_MODELS

class ContentAnalyzer:
    def __init__(self, project_hint):
        self.project_hint = project_hint
        self.domain_context = DOMAIN_CONTEXT
        self.known_models = KNOWN_TS_MODELS

    # Stage 1: Radical Precision Identification (Uses Fast Model)
    def identify_potential_names(self, text):
        \"\"\"Scans text snippets to identify potential dataset/model names.\"\"\"
        PROMPT_IDENTIFY_RADICAL_PRECISION = \"\"\"
        You are an expert analyst identifying ONLY established DATASETS and MODELS/METHODS within a specific domain.

        DOMAIN CONTEXT: {domain_context}
        PROJECT HINT: {project_hint}

        CRITICAL INSTRUCTION: EXTREME PRECISION REQUIRED. Filter ruthlessly.

        1. Analyze the text snippet.
        2. Identify ONLY named entities that are specific, recognized datasets or algorithms relevant to the DOMAIN CONTEXT.

        RULES FOR INCLUSION (Must meet ALL):
        - Must be a proper noun or established acronym (e.g., "ETTh1", "PatchTST", "KDD Cup 2018").
        - Must refer to a collection of data or a specific algorithm.

        RULES FOR EXCLUSION (If ANY met, EXCLUDE):
        - DO NOT INCLUDE generic terms (e.g., "time series", "forecasting", "hourly data", "experiments").
        - DO NOT INCLUDE variables, metrics, or concepts (e.g., "oil temperature", "MAE", "pre-training", "S&P 500").
        - DO NOT INCLUDE organizations or locations unless they are part of the dataset name.

        You must output a JSON list of strings. If none found, output [].

        Text to analyze (Snippet):
        ----------------
        {text_to_analyze}
        \"\"\"
        prompt = PROMPT_IDENTIFY_RADICAL_PRECISION.format(
            domain_context=self.domain_context,
            project_hint=self.project_hint,
            text_to_analyze=text[:15000]
        )
        try:
            # Use fast model for Stage 1
            model = get_model('fast')
            if model is None: return []
            response = generate_content_with_retry(model, prompt)

            try:
                names = json.loads(response.text)
            except json.JSONDecodeError:
                return []

            if isinstance(names, dict):
                names = next((v for v in names.values() if isinstance(v, list)), [])

            if not isinstance(names, list):
                return []

            unique_names = sorted(list(set([n for n in names if isinstance(n, str) and n.strip() and len(n) > 2])))
            return unique_names
        except (RetryError, Exception) as e:
            print(f"    [Warning] API Error during identification step. Skipping snippet.")
            return []

    # Stage 2: Ruthless Classification and Filtering (Uses Accurate Model)
    def classify_filter(self, potential_names, context_preview):
        \"\"\"Classifies identified names and filters out those not used in the project.\"\"\"
        PROMPT_CLASSIFY_FILTER = \"\"\"
        You are an expert data analyst. Filter the Potential Items List using ONLY the Context Preview.

        Project Context: {project_hint}

        CRITICAL INSTRUCTION: RUTHLESS FOCUS.
        We ONLY catalog datasets explicitly USED in the "{project_hint}" project. If the Context Preview does not confirm it was USED, classify as FILTER.

        KNOWN MODELS (Filter these out): {known_models}

        TASKS:
        1. **CLASSIFICATION AND FILTERING (STRICT):**
            - **FILTER:** If it is a MODEL/METHOD, a generic term, OR not confirmed as USED by {project_hint}.
            - **Core (Dataset):** Used in primary experiments/benchmarks.
            - **Pretraining (Dataset):** Used for pretraining.
            - **Novel (Dataset):** Introduced by {project_hint}.

        2. **CANONICALIZATION:** Standardize dataset names.

        Potential Items List: {potential_names}

        You must output a JSON list of objects.
        Example: [{{"name": "ETTh1", "classification": "Core"}}, {{"name": "PatchTST", "classification": "FILTER"}}]

        Context Preview (STRICTLY Project-Specific Information):
        --------------------------------
        {context}
        \"\"\"
        print("Stage 2: Classifying and Filtering (Ruthless Filter)...")

        # Use the accurate model (Pro) for this critical analysis step
        model = get_model('accurate')
        if model is None:
             print("[Error] Accurate model not available. Proceeding with fallback.")
             return [{"name": name, "classification": "Unverified"} for name in potential_names]

        prompt = PROMPT_CLASSIFY_FILTER.format(
            project_hint=self.project_hint,
            potential_names=json.dumps(potential_names),
            known_models=json.dumps(list(self.known_models)),
            context=context_preview[:50000]
        )

        try:
            response = generate_content_with_retry(model, prompt)

            # Robust Parsing
            try:
                classified_items = json.loads(response.text)
            except json.JSONDecodeError:
                return [{"name": name, "classification": "Unverified"} for name in potential_names]

            if isinstance(classified_items, dict):
                classified_items = next((v for v in classified_items.values() if isinstance(v, list)), [])

            if not isinstance(classified_items, list):
                return [{"name": name, "classification": "Unverified"} for name in potential_names]

            # Apply the Filter
            valid_datasets = []
            processed_names = set()

            for item in classified_items:
                if isinstance(item, dict) and 'name' in item and 'classification' in item and item['name'].strip():
                    name = item['name']
                    if name in processed_names: continue
                    processed_names.add(name)

                    if item['classification'].upper() != 'FILTER':
                        valid_datasets.append(item)

            print(f"Analysis complete. Datasets retained: {len(valid_datasets)}.")
            return valid_datasets

        except (RetryError, Exception) as e:
            print(f"[Error] Error during classification/analysis step. Proceeding with fallback.")
            return [{"name": name, "classification": "Unverified"} for name in potential_names]
"""

# --- v40_core/extractor.py ---
V40_CORE_EXTRACTOR_PY = """
# v40_core/extractor.py
# Implements Stage 3 (Precision RAG Extraction).

import json
import time
from tenacity import RetryError
from llama_index.core import VectorStoreIndex

# Import centralized LLM utilities
from v40_tools.llm_utils import get_model, generate_content_with_retry

# Import configuration and schema
from v40_config import DOMAIN_CONTEXT
from v40_core.schema import CATALOG_SCHEMA_DEFINITION

class DataExtractor:
    def __init__(self, project_hint):
        self.project_hint = project_hint
        self.domain_context = DOMAIN_CONTEXT
        self.schema = CATALOG_SCHEMA_DEFINITION

    # Stage 3: Precision RAG (Uses Accurate Model)
    def extract_details(self, index: VectorStoreIndex, dataset_name: str, field_name: str):
        \"\"\"Performs targeted RAG to extract a specific field value and confidence.\"\"\"

        PROMPT_PRECISION_RAG_EXTRACT = \"\"\"
        Extract specific information based ONLY on the provided context snippets.

        Dataset Name: "{dataset_name}"
        Target Information: "{target_field}"
        Specific Question: {specific_query}

        CRITICAL RULES:
        1. **Context Strictness:** Answer STRICTLY based on the context below.
        2. **MANDATORY RANGES:** If the value varies, you MUST provide a specific range (e.g., "Varies (100-500)"). "Varies" alone is FORBIDDEN.
        3. **LINK EXTRACTION:** If the target is a link, extract the RAW URL string.
        4. **SINGLE LINE:** Output must be strictly single-line text.
        5. **HANDLING MISSING DATA:** If the context does not contain the answer, output "[MISSING]".

        You must output a JSON object with two keys:
        - "value": The extracted information (string).
        - "confidence": A float (0.0-1.0) indicating confidence based *only* on the context.

        Retrieved Context Snippets:
        ---------------------------------------
        {context}
        \"\"\"

        if field_name not in self.schema:
            raise ValueError(f"Invalid field name: {field_name}")

        definition = self.schema[field_name]
        base_query = definition["query"].format(name=dataset_name)

        # Contextualized Retrieval Query
        retrieval_context = f"CONTEXT: {self.domain_context}."
        if self.project_hint:
            retrieval_context += f" PROJECT: {self.project_hint}."
        retrieval_query = f"{retrieval_context} QUERY: {base_query}"

        retriever = index.as_retriever(similarity_top_k=8)

        # 1. Retrieval Step
        try:
            retrieved_nodes = retriever.retrieve(retrieval_query)

            # Synthesize context
            retrieved_context = ""
            for node_with_score in retrieved_nodes:
                node = node_with_score.node
                retrieved_context += f"[Source]\\n{node.get_content()}\\n\\n"

            # 2. Synthesis Step (LLM - Targeted)
            prompt = PROMPT_PRECISION_RAG_EXTRACT.format(
                dataset_name=dataset_name,
                target_field=field_name,
                specific_query=json.dumps(base_query),
                context=retrieved_context[:30000]
            )

            # Use the accurate model for synthesis
            model = get_model('accurate')
            if model is None:
                 return {"value": "[MISSING]", "confidence": 0.0}

            response = generate_content_with_retry(model, prompt)

            try:
                result = json.loads(response.text)
                value = result.get("value", "[MISSING]")
                confidence = float(result.get("confidence", 0.0))
            except (json.JSONDecodeError, ValueError, TypeError):
                value = "[MISSING]"
                confidence = 0.0

            time.sleep(0.5)
            return {"value": value, "confidence": confidence}

        except (RetryError, Exception) as e:
            print(f"    [Error] Error during Precision RAG for {dataset_name}/{field_name}.")
            return {"value": "[MISSING]", "confidence": 0.0}
"""

# --- v40_core/orchestrator.py ---
# This contains the optimized V39 engine logic (Iterative Refinement, Caching, Locking)
V40_CORE_ORCHESTRATOR_PY = """
# v40_core/orchestrator.py
# The main pipeline logic, managing the iterative loop, state, caching, and auditing.

import time
import hashlib
import pandas as pd
import re
from llama_index.core import VectorStoreIndex, Document, Settings

# Import configuration, schema, and core components
from v40_config import (
    MAX_RESEARCH_ITERATIONS, CONFIDENCE_THRESHOLD, CONFIDENCE_LOCK_THRESHOLD
)
from v40_core.schema import (
    CATALOG_SCHEMA_DEFINITION, MISSING_DATA_PLACEHOLDERS, AMBIGUOUS_DATASETS
)
from v40_core.analyzer import ContentAnalyzer
from v40_core.extractor import DataExtractor

class ResearchOrchestrator:
    \"\"\"Orchestrates the agentic research plan using the V39 optimized engine.\"\"\"

    def __init__(self, loader, search_tool, kb_tool, project_hint):
        self.loader = loader
        self.search_tool = search_tool
        self.kb_tool = kb_tool
        self.project_hint = project_hint

        # Initialize core components
        self.analyzer = ContentAnalyzer(project_hint)
        self.extractor = DataExtractor(project_hint)

        # State management
        self.corpus_documents = []
        self.loaded_urls = set()
        self.index = None
        # V39 Optimization: Track documents processed by Stage 1
        self.stage1_processed_doc_hashes = set()

        self.schema = CATALOG_SCHEMA_DEFINITION

    def run(self, input_urls, max_iterations=MAX_RESEARCH_ITERATIONS):
        print(f"\\n=== DeepCollector Research Pipeline (V40 UDR-Arch, V39 Engine) ===")
        print(f"Target: {self.project_hint}")

        # Phase 1 & 2: Discovery and Exploration
        print("\\n--- Phase 1 & 2: Discovery and Exploration ---")
        self._initial_discovery(input_urls)
        self._exploratory_search()

        # Phase 3: Iterative Research Loop
        print("\\n--- Phase 3: Iterative Research Loop ---")
        final_catalog_data = self._research_loop(max_iterations)

        # Phase 4: Post-Processing and Formatting
        print("\\n--- Phase 4: Post-Processing and Formatting ---")
        final_catalog_df = self._format_catalog(final_catalog_data)

        return final_catalog_df

    # --- Phase 1 & 2 Helpers (Discovery/Exploration) ---

    def _add_documents_to_corpus(self, documents):
        if documents:
            self.corpus_documents.extend(documents)
            if self.index:
                try:
                    nodes = Settings.node_parser(documents)
                    self.index.insert_nodes(nodes)
                except Exception as e:
                    print(f"\\n*** WARNING: Failed to update Vector Index. Error: {e} ***")

    def _get_corpus_text_preview(self):
        preview = ""
        MAX_PREVIEW_DOCS = 15
        for doc in self.corpus_documents[:MAX_PREVIEW_DOCS]:
            source = doc.metadata.get("source_url", "Unknown Source")
            preview += f"\\n\\n<<< START SOURCE: {source} >>>\\n\\n"
            preview += doc.get_content()[:800] + "..."
            preview += f"\\n\\n<<< END SOURCE: {source} >>>\\n\\n"
        return preview

    def _initial_discovery(self, input_urls):
        self._load_content(input_urls)
        # Note: Supplementary link discovery is often handled implicitly by the analyzer/search now.

    def _load_content(self, urls):
        MAX_LOADS = 15
        loads = 0
        loaded_urls = []
        if isinstance(urls, str): urls = [urls]
        if not urls: return loaded_urls

        for url in urls:
            if loads >= MAX_LOADS: break
            if url not in self.loaded_urls:
                documents = self.loader.load(url)
                self.loaded_urls.add(url)
                if documents:
                    self._add_documents_to_corpus(documents)
                    loaded_urls.append(url)
                    loads += 1
        return loaded_urls

    def _exploratory_search(self):
        if not self.search_tool or not self.project_hint: return

        base_context = "time series" # Domain context enforcement
        queries = [
            f'"{self.project_hint}" {base_context} datasets used experiments',
            f'"{self.project_hint}" {base_context} pretraining datasets',
            f'"{self.project_hint}" (site:huggingface.co OR site:paperswithcode.com OR site:github.com)'
        ]

        FETCH_LIMIT = 2

        for query in queries:
            try:
                search_results = self.search_tool.search(query=query, num_results=8)
                urls_to_fetch = []
                snippet_documents = []

                for result in search_results:
                    url = result.get('url', 'N/A')
                    if url not in self.loaded_urls and url not in urls_to_fetch:
                        urls_to_fetch.append(url)
                    snippet_text = f"Title: {result.get('source_title')}\\nURL: {url}\\nSnippet: {result.get('snippet')}"
                    snippet_documents.append(Document(text=snippet_text, metadata={"source_url": f"[EXPLORATORY_SNIPPET]"}))

                self._add_documents_to_corpus(snippet_documents)

                if urls_to_fetch:
                    self._load_content(urls_to_fetch[:FETCH_LIMIT])

                time.sleep(1)
            except Exception as e:
                print(f"    Error during exploratory search: {e}")

    # --- Phase 3: The Research Loop (V39 Optimized Engine) ---

    def _research_loop(self, max_iterations):
        # State persists across iterations
        current_catalog_data = []
        gaps = []

        # Initialize the index
        if self.index is None and self.corpus_documents:
            print(f"Initializing Vector Index for RAG ({len(self.corpus_documents)} documents)...")
            try:
                self.index = VectorStoreIndex.from_documents(self.corpus_documents)
            except Exception as e:
                print(f"\\n*** CRITICAL ERROR: Failed to initialize Vector Index. Error: {e} ***")
                return []
        elif not self.corpus_documents:
           return []

        for iteration in range(max_iterations):
            print(f"\\n--- Research Loop Iteration {iteration + 1}/{max_iterations} ---")

            # 1. Extraction/Refinement (Targeted updates based on gaps)
            current_catalog_data = self._execute_extraction_pipeline(current_catalog_data, gaps, iteration)

            # 2. Audit (Identify new gaps)
            gaps = self._audit_catalog(current_catalog_data)

            if not gaps:
                print(f"Audit Passed: All cells meet confidence threshold (>={CONFIDENCE_THRESHOLD}) or are verified.")
                return current_catalog_data

            print(f"Audit Identified Gaps: {len(gaps)} specific fields need refinement.")

            # 3. Targeted Search
            if not self.search_tool:
                print("Search tool unavailable. Stopping loop.")
                return current_catalog_data

            new_information_found = self._targeted_search(gaps)

            if not new_information_found:
                if iteration > 0 and len(current_catalog_data) > 0:
                     print("Targeted search yielded no new information. Stopping loop.")
                     return current_catalog_data

            time.sleep(2)

        print("Maximum iterations reached. Returning best effort catalog.")
        return current_catalog_data

    # The Optimized Extraction Pipeline (V39 Engine Logic)
    def _execute_extraction_pipeline(self, current_catalog_data, gaps, iteration):
        \"\"\"
        The core V39 engine: optimized, iterative extraction pipeline.
        \"\"\"

        # --- Stage 1: Identify (Optimized Incremental Scan) ---
        potential_names = set()
        print("Stage 1: Identifying potential entities (Optimized Scan)...")

        # Identify new documents using robust hashing
        new_documents = []
        new_document_hashes = set()

        for doc in self.corpus_documents:
            doc_hash = getattr(doc, 'hash', None)
            if not doc_hash:
                 try:
                    doc_hash = hashlib.sha256(doc.get_content().encode('utf-8')).hexdigest()
                 except Exception:
                    doc_hash = f"fallback_id_{doc.id_}"

            if doc_hash not in self.stage1_processed_doc_hashes:
                new_documents.append(doc)
                new_document_hashes.add(doc_hash)

        if new_documents or iteration == 0:
            print(f"    Scanning {len(new_documents)} new documents...")
            for doc in new_documents:
                content = doc.get_content()
                if not content: continue

                # V40: Use the Analyzer component
                names = self.analyzer.identify_potential_names(content)
                potential_names.update(names)

            self.stage1_processed_doc_hashes.update(new_document_hashes)
        else:
            print("    No new documents to scan.")

        # --- Stage 2: Classify, Filter (Conditional Execution) ---
        if potential_names or iteration == 0:
            print(f"    New potential entities identified: {len(potential_names)}. Proceeding to classification.")
            existing_names = {entry.get('Dataset Name') for entry in current_catalog_data if entry.get('Dataset Name')}
            all_names = list(potential_names.union(existing_names))

            if all_names:
                context_preview = self._get_corpus_text_preview()
                # V40: Use the Analyzer component (Expensive call)
                classified_datasets = self.analyzer.classify_filter(all_names, context_preview)

                # Determine if the catalog structure has changed
                new_classified_names = {item['name'] for item in classified_datasets}
                if new_classified_names != existing_names:
                    print("    Catalog structure changed. Rebuilding base catalog data.")
                    # Merge new classifications with existing data
                    new_catalog_data = []
                    existing_data_map = {entry['Dataset Name']: entry for entry in current_catalog_data if entry.get('Dataset Name')}

                    for item in classified_datasets:
                        name = item['name']
                        if name in existing_data_map:
                            entry = existing_data_map[name]
                            entry['Classification'] = item['classification'] # Update classification
                            new_catalog_data.append(entry)
                        else:
                            # New dataset found
                            new_catalog_data.append({"Dataset Name": name, "Classification": item['classification']})
                    current_catalog_data = new_catalog_data
                else:
                    print("    Catalog structure unchanged.")
        else:
            print("Stage 2: Skipped (No new entities identified).")

        if not current_catalog_data:
            return []

        # --- Stage 3: Enrich Details (Targeted RAG and Confidence Locking) ---
        self._enrich_details_targeted(current_catalog_data, gaps)

        return current_catalog_data

    # Stage 3 Implementation (V39 Optimization: Targeted Updates and Confidence Locking)
    def _enrich_details_targeted(self, catalog_data, gaps):
        \"\"\"Updates catalog_data in place, processing only gaps or unlocked cells.\"\"\"

        print("Stage 3: Retrieving details (V40: Targeted Precision RAG)...")

        # Create a set of (dataset, field) tuples that need processing
        required_updates = set()
        for gap in gaps:
            if gap['dataset'] != "General Context":
                required_updates.add((gap['dataset'], gap['field']))

        total_cells_processed = 0
        total_cells_skipped = 0

        # Iterate through the existing catalog data
        for entry in catalog_data:
            name = entry.get('Dataset Name')
            if not name: continue

            # Iterate through each field defined in the schema
            for field, definition in self.schema.items():
                cell_data = entry.get(field)

                # --- Optimization Logic ---
                needs_processing = False

                # Condition 1: Field data is missing (handles new datasets)
                if not isinstance(cell_data, dict) or 'confidence' not in cell_data:
                    needs_processing = True
                else:
                    confidence = cell_data.get("confidence", 0.0)
                    # Condition 2: Confidence is very high (lock and skip)
                    if confidence >= CONFIDENCE_LOCK_THRESHOLD:
                        total_cells_skipped += 1
                        continue
                    # Condition 3: Field was explicitly listed in the gaps
                    elif (name, field) in required_updates:
                        needs_processing = True
                    # Condition 4: If gaps list was empty (e.g., first iteration), process everything below lock
                    elif not gaps:
                        needs_processing = True

                if not needs_processing:
                    total_cells_skipped += 1
                    continue
                # --------------------------------

                # Proceed with RAG retrieval and synthesis
                total_cells_processed += 1

                # V40: Use the Extractor component
                result = self.extractor.extract_details(self.index, name, field)
                entry[field] = result

            # 3. Knowledge Base Tool Lookup (Fallback)
            if self.kb_tool:
                self._apply_kb_fallback(entry, name)

        print(f"Stage 3 complete. Cells Processed (LLM Calls): {total_cells_processed}. Cells Skipped (Cached/Locked): {total_cells_skipped}.")

    def _apply_kb_fallback(self, entry, name):
        \"\"\"Applies KB data if RAG results are missing or low confidence.\"\"\"
        kb_entry = self.kb_tool.lookup(name)
        if kb_entry:
            kb_mapping = {
                'Original Source or Originator of Data': ['Original Source', 'Originator', 'Primary Source Repository', 'Link', 'URL'],
                'Link to Data': ['Link to Data', 'Download Link', 'Link', 'URL'],
                'Domain': ['Domain', 'Area'],
                'Number of Variables at each time point': ['Variables', '#Variables', 'Dimensionality'],
                'Number of Time Points': ['Time Points', '#Time Points', 'Length'],
                'Time interval between points': ['Frequency', 'Interval']
            }

            for field, keys in kb_mapping.items():
                cell_data = entry.get(field)
                if isinstance(cell_data, dict):
                    current_value = str(cell_data.get("value", "")).strip().lower()
                    current_confidence = cell_data.get("confidence", 0.0)

                    # Check if KB fallback is needed
                    if current_confidence < CONFIDENCE_THRESHOLD and current_value in MISSING_DATA_PLACEHOLDERS:
                        kb_value = self.kb_tool.extract_kb_value(kb_entry, keys)
                        if kb_value:
                            # Apply KB value and mark confidence as 1.0 (Verified)
                            entry[field] = {"value": kb_value, "confidence": 1.0}

    # Audit Logic (Unchanged from V39)
    def _audit_catalog(self, catalog_data):
        \"\"\"Audits the catalog based on individual cell confidence and completeness.\"\"\"
        gaps = []
        if not catalog_data:
             if self.search_tool:
                return [{"dataset": "General Context", "field": "All", "keywords": "find all datasets mentioned", "classification": "General"}]
             else:
                return []

        for entry in catalog_data:
            dataset_name = entry.get('Dataset Name')
            classification = entry.get('Classification', 'Unknown')

            if not dataset_name: continue

            for field, definition in self.schema.items():
                cell_data = entry.get(field)

                if not isinstance(cell_data, dict):
                    gaps.append({"dataset": dataset_name, "field": field, "keywords": definition["keywords"], "classification": classification})
                    continue

                value = str(cell_data.get("value", "")).strip().lower()
                confidence = cell_data.get("confidence", 0.0)

                is_missing = pd.isna(value) or value in MISSING_DATA_PLACEHOLDERS
                is_low_confidence = confidence < CONFIDENCE_THRESHOLD
                is_ambiguous = value == "varies"

                # Determine if a gap exists
                if is_missing or (is_low_confidence and not is_missing):
                    search_keywords = definition["keywords"]
                    if not is_missing:
                        search_keywords = f"verify {search_keywords}"
                    gaps.append({"dataset": dataset_name, "field": field, "keywords": search_keywords, "classification": classification})

                elif is_ambiguous:
                    search_keywords = f"estimate specific range for {definition['keywords']}"
                    gaps.append({"dataset": dataset_name, "field": field, "keywords": search_keywords, "classification": classification})

        return gaps

    # Targeted Search (Unchanged from V39)
    def _targeted_search(self, gaps):
        MAX_SEARCHES = 12
        MAX_DEEP_FETCHES = 5
        deep_fetches_performed = 0
        searches_performed = 0
        new_information_found = False

        for gap in gaps:
            if searches_performed >= MAX_SEARCHES: break

            dataset = gap['dataset']
            field = gap['field']
            keywords = gap['keywords']
            classification = gap.get('classification', 'Unknown')

            # Strategy Selection
            is_ambiguous_name = dataset.lower() in AMBIGUOUS_DATASETS
            is_complex_dataset = classification in ["Novel", "Ancillary", "Pretraining", "General"]
            base_context = "time series dataset"

            # Strategy 1: Context-Strict Search
            if is_ambiguous_name or is_complex_dataset:
                context_terms = base_context
                if self.project_hint and dataset != "General Context":
                    context_terms += f' used in "{self.project_hint}"'
                query = f'"{dataset}" {context_terms} {keywords}' if dataset != "General Context" else keywords
            # Strategy 2: Standard Targeted Search
            else:
                query = f'"{dataset}" {base_context} {keywords}'

            # Execution
            try:
                search_results = self.search_tool.search(query=query, num_results=8)
                searches_performed += 1

                urls_to_fetch = []
                snippet_documents = []

                for result in search_results:
                    url = result.get('url', 'N/A')
                    if url not in self.loaded_urls and url not in urls_to_fetch:
                        urls_to_fetch.append(url)

                    snippet_text = f"Title: {result.get('source_title')}\\nURL: {url}\\nSnippet: {result.get('snippet')}"
                    snippet_documents.append(Document(text=snippet_text, metadata={"source_url": f"[TARGETED_SNIPPET]_{dataset}_{field}"}))

                if snippet_documents:
                    self._add_documents_to_corpus(snippet_documents)
                    new_information_found = True

                    # Deep Search Analysis (Fetching content)
                    needs_deep_fetch = "link" in keywords or "range" in keywords or "verify" in keywords

                    if deep_fetches_performed < MAX_DEEP_FETCHES and urls_to_fetch and needs_deep_fetch:
                        fetch_limit = 2 if is_ambiguous_name else 1
                        fetched_urls = self._load_content(urls_to_fetch[:fetch_limit])
                        deep_fetches_performed += len(fetched_urls)

            except Exception as e:
                print(f"    Error during targeted search: {e}")

            time.sleep(1)

        return new_information_found

    # --- Phase 4: Post-Processing ---

    def _format_catalog(self, catalog_data):
        \"\"\"Converts the raw data structure into a flat DataFrame and applies formatting.\"\"\"
        if not catalog_data:
            return pd.DataFrame()

        print("Formatting catalog and applying HYPERLINK strategy...")

        # 1. Flatten the data structure
        flat_data = []
        for entry in catalog_data:
            flat_entry = {}
            flat_entry["Dataset Name"] = entry.get("Dataset Name")
            flat_entry["_Classification"] = entry.get("Classification")

            for field, definition in self.schema.items():
                cell_data = entry.get(field)
                if isinstance(cell_data, dict):
                    flat_entry[field] = cell_data.get("value", "[MISSING]")
                else:
                    flat_entry[field] = "[MISSING]"

            flat_data.append(flat_entry)

        df = pd.DataFrame(flat_data)
        df = df.fillna('')

        # 2. Apply HYPERLINK formatting
        def format_link(value, display_text_hint="Link"):
            if not isinstance(value, str): return value
            value = value.strip()
            if not value or value.lower() in MISSING_DATA_PLACEHOLDERS or value.lower().startswith('=hyperlink('):
                return value

            potential_urls = re.findall(r'https?://[^\\s;]+', value)
            if not potential_urls: return value

            cleaned_urls = []
            for url in potential_urls:
                cleaned_url = re.sub(r'[.,)}\\]>]+$', '', url)
                cleaned_urls.append(cleaned_url)

            unique_urls = list(dict.fromkeys(cleaned_urls))

            if len(unique_urls) == 1:
                if len(value) <= len(unique_urls[0]) + 15:
                    return f'=HYPERLINK("{unique_urls[0]}", "{display_text_hint}")'
                else:
                    return value

            # Multiple Links
            formatted_string = ""
            link_counter = 1
            for url in unique_urls:
                display_text = f"{display_text_hint} {link_counter}"
                if formatted_string: formatted_string += "; "
                formatted_string += f"{display_text}: {url}"
                link_counter += 1
            return formatted_string

        if 'Original Source or Originator of Data' in df.columns:
            df['Original Source or Originator of Data'] = df['Original Source or Originator of Data'].apply(lambda x: format_link(x, "Originator"))

        if 'Link to Data' in df.columns:
            df['Link to Data'] = df['Link to Data'].apply(lambda x: format_link(x, "Link"))

        # Add Comments column
        if '_Classification' in df.columns:
            df['Comments'] = df['_Classification'].apply(lambda x: f"Classification: {x}." if x else "")

        # 3. Drop internal columns
        if '_Classification' in df.columns:
            df = df.drop(columns=['_Classification'])

        # Final cleanup
        df = df.replace(r'\\n',' ', regex=True).replace(r'\\r', ' ', regex=True)

        return df
"""

# ====================================================================================
# MAIN EXECUTION
# ====================================================================================

# --- v40_main.py ---
V40_MAIN_PY = """
# v40_main.py
# Execution and Output Handling for DeepCollector V40.

import time
import pandas as pd
from IPython.display import display
import gspread
from tenacity import RetryError

# Import configuration and initialization
from v40_config import (
    initialize_environment, GSPREAD_RETRY_STRATEGY,
    PRIMARY_URLS, PROJECT_NAME_HINT, SPREADSHEET_NAME, GENERAL_KNOWLEDGE_PKL_PATH
)
# Import auth separately as they might be None if not in Colab
from v40_config import auth, default

# Import tools
from v40_tools.llm_utils import initialize_models
from v40_tools.loader import DocumentLoader
from v40_tools.search import SearchTool
from v40_tools.kb import KnowledgeBaseTool

# Import core orchestrator
from v40_core.orchestrator import ResearchOrchestrator
from v40_core.schema import MISSING_DATA_PLACEHOLDERS

# ------------------------------------------------------
# Helper Function: Save to Google Sheets
# ------------------------------------------------------

def save_to_google_sheets_robust(df, spreadsheet_name):
    \"\"\"Authenticates the user and saves the DataFrame using native gspread with robust retries.\"\"\"
    if df.empty: return

    # Clean the DataFrame before uploading
    try:
        if hasattr(pd.DataFrame, 'map'):
            df_cleaned = df.map(lambda x: '' if str(x).strip().lower() in MISSING_DATA_PLACEHOLDERS else x)
        else:
            df_cleaned = df.applymap(lambda x: '' if str(x).strip().lower() in MISSING_DATA_PLACEHOLDERS else x)
    except Exception as e:
        print(f"Warning: Data cleaning failed: {e}.")
        df_cleaned = df.copy()

    # Ensure single line constraint
    df_cleaned = df_cleaned.replace(r'\\n',' ', regex=True).replace(r'\\r', ' ', regex=True)

    print(f"\\n--- Google Sheets Authorization (for Export) ---")

    # AUTHENTICATION STEP
    try:
        if not auth or not default:
            raise ImportError("Google Colab auth libraries not loaded.")

        print("CRITICAL: A prompt may appear below asking for authorization for Google Sheets.")
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        print("Authorization successful.")
    except Exception as e:
        print(f"\\n*** AUTHENTICATION FAILED: {e}. ***")
        return

    print("\\n--- Uploading Data (Robust Method) ---")

    # Define wrapper functions for GSpread operations
    @GSPREAD_RETRY_STRATEGY
    def open_spreadsheet(gc, name):
        print(f"  Attempting to open/create spreadsheet: {name}...")
        try:
            sh = gc.open(name)
            print(f"  Opened existing spreadsheet.")
            return sh
        except gspread.SpreadsheetNotFound:
            sh = gc.create(name)
            print(f"  Created new spreadsheet.")
            return sh

    @GSPREAD_RETRY_STRATEGY
    def upload_data(worksheet, data):
        print("  Attempting to upload data...")
        worksheet.clear()
        # CRITICAL: value_input_option='USER_ENTERED' for formulas (HYPERLINK).
        worksheet.update('A1', data, value_input_option='USER_ENTERED')
        print("  Upload successful.")

    @GSPREAD_RETRY_STRATEGY
    def resize_columns(sh, worksheet, num_columns):
        print("  Attempting to resize columns...")
        sh.batch_update({
            "requests": [{"autoResizeDimensions": {
                "dimensions": {
                    "sheetId": worksheet.id, "dimension": "COLUMNS",
                    "startIndex": 0, "endIndex": num_columns
                }}}]})

    try:
        # Execute the operations with retries
        sh = open_spreadsheet(gc, spreadsheet_name)
        worksheet = sh.get_worksheet(0)

        # Prepare data for upload
        data_to_upload = [df_cleaned.columns.values.tolist()] + df_cleaned.astype(str).values.tolist()
        upload_data(worksheet, data_to_upload)

        # Basic formatting
        try:
            resize_columns(sh, worksheet, len(df_cleaned.columns))
        except Exception:
            pass

        print(f"\\nSuccessfully finalized data export.")
        print(f"Link to Sheet: {sh.url}")

    except RetryError as e:
        print(f"\\n*** CRITICAL ERROR: Failed to save to Google Sheets after multiple attempts. ***")
    except Exception as e:
        print(f"\\nAn unexpected error occurred while saving to Google Sheets: {e}")

# ------------------------------------------------------
# Main Execution Function (V40)
# ------------------------------------------------------

def run_pipeline_v40():
    \"\"\"Initializes the environment, runs the pipeline, and handles output.\"\"\"

    # 1. Initialize environment and get configuration
    config = initialize_environment()

    # 2. Initialize Models (Centralized in llm_utils)
    initialize_models(config.get("GEMINI_API_KEY"))

    # 3. Initialize Tools
    loader = DocumentLoader()

    if config.get("SEARCH_ENABLED"):
        search_tool_instance = SearchTool(config["GOOGLE_SEARCH_API_KEY"], config["GOOGLE_CSE_ID"])
    else:
        search_tool_instance = None

    print("\\n--- Initializing KnowledgeBaseTool ---")
    kb_tool_instance = KnowledgeBaseTool(GENERAL_KNOWLEDGE_PKL_PATH)

    # 4. Initialize Orchestrator
    orchestrator = ResearchOrchestrator(
        loader,
        search_tool_instance,
        kb_tool=kb_tool_instance,
        project_hint=PROJECT_NAME_HINT
    )

    # 5. Run the pipeline
    start_time = time.time()
    df_catalog = orchestrator.run(PRIMARY_URLS)
    end_time = time.time()

    print(f"\\nTotal execution time: {(end_time - start_time):.2f} seconds.")

    # 6. Display Results and Export
    if isinstance(df_catalog, pd.DataFrame) and not df_catalog.empty:
        print("\\n--- Final Catalog ---")
        pd.set_option('display.max_colwidth', None)
        try:
            # Displaying the catalog cleaned of internal placeholders
            if hasattr(pd.DataFrame, 'map'):
                display(df_catalog.map(lambda x: '' if str(x).strip().lower() == '[missing]' else x))
            else:
                display(df_catalog.applymap(lambda x: '' if str(x).strip().lower() == '[missing]' else x))
        except Exception:
            display(df_catalog)

        # Save to Google Sheets
        save_to_google_sheets_robust(df_catalog, SPREADSHEET_NAME)

    else:
        print("\\nPipeline did not produce a catalog.")
"""

# ------------------------------------------------------------------------------------
# 3. Write Module Files Dynamically
# ------------------------------------------------------------------------------------
print("\n--- Writing Module Files (V40 Architecture) ---")
try:
    # Clean up previous versions if they exist
    if os.path.exists("v40_tools"):
        shutil.rmtree("v40_tools")
    if os.path.exists("v40_core"):
        shutil.rmtree("v40_core")

    # Create directories
    os.makedirs("v40_tools", exist_ok=True)
    os.makedirs("v40_core", exist_ok=True)

    # Write __init__.py files
    with open("v40_tools/__init__.py", "w") as f: f.write("")
    with open("v40_core/__init__.py", "w") as f: f.write("")

    # Write Configuration
    with open("v40_config.py", "w") as f:
        f.write(V40_CONFIG_PY)
    print("Wrote v40_config.py")

    # Write Tools
    with open("v40_tools/llm_utils.py", "w") as f:
        f.write(V40_TOOLS_LLM_UTILS_PY)
    print("Wrote v40_tools/llm_utils.py")
    with open("v40_tools/search.py", "w") as f:
        f.write(V40_TOOLS_SEARCH_PY)
    print("Wrote v40_tools/search.py")
    with open("v40_tools/loader.py", "w") as f:
        f.write(V40_TOOLS_LOADER_PY)
    print("Wrote v40_tools/loader.py")
    with open("v40_tools/kb.py", "w") as f:
        f.write(V40_TOOLS_KB_PY)
    print("Wrote v40_tools/kb.py")

    # Write Core
    with open("v40_core/schema.py", "w") as f:
        f.write(V40_CORE_SCHEMA_PY)
    print("Wrote v40_core/schema.py")
    with open("v40_core/analyzer.py", "w") as f:
        f.write(V40_CORE_ANALYZER_PY)
    print("Wrote v40_core/analyzer.py")
    with open("v40_core/extractor.py", "w") as f:
        f.write(V40_CORE_EXTRACTOR_PY)
    print("Wrote v40_core/extractor.py")
    with open("v40_core/orchestrator.py", "w") as f:
        f.write(V40_CORE_ORCHESTRATOR_PY)
    print("Wrote v40_core/orchestrator.py")

    # Write Main
    with open("v40_main.py", "w") as f:
        f.write(V40_MAIN_PY)
    print("Wrote v40_main.py")

except IOError as e:
    print(f"*** ERROR writing module files: {e} ***")

# ------------------------------------------------------------------------------------
# 4. Execution Block
# ------------------------------------------------------------------------------------
print("\n--- Executing Pipeline ---")

# Add current directory to path
if '.' not in sys.path:
    sys.path.append('.')

# Attempt to import and execute
try:
    # Import modules (necessary to ensure the newly written files are recognized)
    import v40_main, v40_config
    # We must import the packages so the modules inside them can be referenced
    from v40_tools import llm_utils, search, loader, kb
    from v40_core import schema, analyzer, extractor, orchestrator

    # Force reload modules (Crucial for iterative development in Colab)
    importlib.reload(v40_config)
    importlib.reload(llm_utils)
    importlib.reload(search)
    importlib.reload(loader)
    importlib.reload(kb)
    importlib.reload(schema)
    importlib.reload(analyzer)
    importlib.reload(extractor)
    importlib.reload(orchestrator)
    importlib.reload(v40_main)

    # Execute the main function
    from v40_main import run_pipeline_v40
    run_pipeline_v40()

except ImportError as e:
    print(f"\n*** CRITICAL ERROR: Failed to import modules after writing: {e} ***")
    print("Please ensure dependencies installed correctly and the environment allows file writing.")
    traceback.print_exc()
except Exception as e:
    print(f"\n*** Error during execution: {e} ***")
    traceback.print_exc()

--- Installing/Upgrading Dependencies ---
Dependencies installed successfully.

--- Writing Module Files (V40 Architecture) ---
Wrote v40_config.py
Wrote v40_tools/llm_utils.py
Wrote v40_tools/search.py
Wrote v40_tools/loader.py
Wrote v40_tools/kb.py
Wrote v40_core/schema.py
Wrote v40_core/analyzer.py
Wrote v40_core/extractor.py
Wrote v40_core/orchestrator.py
Wrote v40_main.py

--- Executing Pipeline ---

--- Google Drive Authorization (for Knowledge Base Access) ---
CRITICAL: A prompt may appear requesting access to Google Drive.
Mounted at /content/drive
Google Drive accessible.

--- API Configuration Check ---
Gemini API configured.
Google Search API configured. Dynamic Discovery ENABLED.
Initialized Gemini Control Flow and LlamaIndex RAG components.

--- Initializing KnowledgeBaseTool ---
KnowledgeBaseTool initialized with 207 entries.

=== DeepCollector Research Pipeline (V40 UDR-Arch, V39 Engine) ===
Target: TEMPO

--- Phase 1 & 2: Discovery and Exploration ---

--- Phase 3: Iter

,Dataset Name,Domain,Number of Variables at each time point,Number of Time Points,Time interval between points,Original Source or Originator of Data,Link to Data,Detailed Description,Comments
0,ECL,"This dataset encompasses various domains including electricity, traffic, and weather, as indicated by the provided context. It also includes a TETS dataset derived from the S&P 500, focusing on financial data, specifically EBITDA of companies.",96,26304,1 hour,"Defu Cao, Furong Jia, Sercan O Arik, Tomas Pfister, Yixiang Zheng, Wen Ye, and Yan Liu at USC",Link 1: https://drive.google.com/drive/folders/13Cg1KYOlzM5C7K8gK8NfC-F3EYxkM3D2?usp=sharing; Link 2: https://pan.baidu.com/s/1r3KhGd0Q9PJIUZdfEYoymg?pwd=i9iy,"ECL is a multivariate time series dataset used in the evaluation of the TEMPO model for zero-shot and self-supervised forecasting, demonstrating strong performance compared to baseline models, especially in self-supervised representation learning where it achieves lower MSE and MAE.",Classification: Core (Dataset).
1,Traffic,Time Series Forecasting,500,17544,1 hour,DC-research,Link 1: https://drive.google.com/drive/folders/13Cg1KYOlzM5C7K8gK8NfC-F3EYxkM3D2?usp=sharing; Link 2: https://pan.baidu.com/s/1r3KhGd0Q9PJIUZdfEYoymg?pwd=i9iy; Link 3: https://huggingface.co/Melady/TEMPO,"The Traffic dataset is used for time series forecasting experiments within the TEMPO framework, often compared against ARIMA models and sometimes used in pre-training, with results presented in terms of MSE/MAE scores.",Classification: Core (Dataset).
2,Weather,Time Series Forecasting,,52696,10 min,The dataset was created by the authors of the TEMPO model.,"=HYPERLINK(""https://drive.google.com/drive/folders/13Cg1KYOlzM5C7K8gK8NfC-F3EYxkM3D2?usp=sharing"", ""Link"")","TETS is a time series benchmark dataset based on the S&P 500 dataset, combining time series data with contextual information obtained from ChatGPT API by querying company information and yearly context, capped at 110 tokens.",Classification: Core (Dataset).
3,ETTm1,Time series,7,69680,15 min,,"=HYPERLINK(""https://drive.google.com/drive/folders/13Cg1KYOlzM5C7K8gK8NfC-F3EYxkM3D2?usp=sharing"", ""Link"")","ETTm1 is a multivariate time series dataset with a length of 69,680 data points, 7 covariates, and a sampling period of 15 minutes.",Classification: Core (Dataset).
4,ETTm2,,7,69680,15 min,,Link 1: https://drive.google.com/drive/folders/13Cg1KYOlzM5C7K8gK8NfC-F3EYxkM3D2?usp=sharing; Link 2: https://pan.baidu.com/s/1r3KhGd0Q9PJIUZdfEYoymg?pwd=i9iy,"ETTm2 is a multivariate time series dataset with a length of 69680, 7 covariates, and a sampling period of 15 minutes.",Classification: Core (Dataset).
5,ETTh1,,7,17420,Hourly,,Link 1: https://drive.google.com/drive/folders/13Cg1KYOlzM5C7K8gK8NfC-F3EYxkM3D2?usp=sharing; Link 2: https://pan.baidu.com/s/1r3KhGd0Q9PJIUZdfEYoymg?pwd=i9iy,ETTh1 is an hourly time series dataset with 17420 data points and 7 covariates.,Classification: Core (Dataset).
6,ETTh2,,7,8545,1 hour,,Link 1: https://drive.google.com/drive/folders/13Cg1KYOlzM5C7K8gK8NfC-F3EYxkM3D2?usp=sharing; Link 2: https://pan.baidu.com/s/1r3KhGd0Q9PJIUZdfEYoymg?pwd=i9iy,"ETTh2 is an hourly-level dataset, part of the ETT (Electricity Transformer Temperature) dataset collection used for time series forecasting benchmarks, potentially containing multiple covariates and used in zero-shot forecasting experiments.",Classification: Core (Dataset).
7,TETS,Finance,,,15 minutes,The TETS dataset is derived from the S&P 500 dataset and utilizes prompts generated via the OPENAI ChatGPT-3.5 API to create textual information corresponding to the time series data. Processed data with text embedding from GPT2 is also included.,"=HYPERLINK(""https://drive.google.com/file/d/1Hu2KFj0kp4kIIpjbss2ciLCV_KiBreoJ/view?usp=drive_link"", ""Link"")","TETS is a multimodal time series dataset based on the S&P 500, combining time series data with contextual information generated by OpenAI's ChatGPT-3.5 API from prompts ba


--- Google Sheets Authorization (for Export) ---
CRITICAL: A prompt may appear below asking for authorization for Google Sheets.
